# Fine-tune LLMs with GRPO

Example: to train a foundation `Qwen2.5-0.5B` for solving maths problems

In [ ]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth" "trl>=0.14" "peft>=0.13" "transformers>=4.45" accelerate datasets bitsandbytes unsloth_zoo

  Cloning https://github.com/unslothai/unsloth to /tmp/pip-install-3o5d8p_i/unsloth_7b6f2a5cd5814520b91a5d2b7f993fc9
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth /tmp/pip-install-3o5d8p_i/unsloth_7b6f2a5cd5814520b91a5d2b7f993fc9
  Resolved https://github.com/unslothai/unsloth to commit b66d2a4c8912828337407218d6afd655bf0b93ca
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.9.2-py3-none-any.whl size=43780552 sha256=827f13af9a3a2ac2a99b6099f7b13e712f8d4df5edc692d65e8be17d89cd652d
  Stored in directory: /tmp/pip-ephem-wheel-cache-3zxlks2q/wheels/94/bc/64/a1c9ffd137a50b590a95fb24d69298c91b05afbf40b5cfe283
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

This block sets up the core components for model loading and fine-tuning:
*   **Imports**: Essential libraries like `gc` (garbage collector), `torch` (PyTorch), `datasets`, and `unsloth` are imported.
*   **Global Variables**: `MODEL_ID` specifies the base model to be used (Qwen/Qwen2.5-0.5B-Instruct), `MAX_SEQ_LEN` defines the maximum sequence length for the model, and `USE_4BIT` enables 4-bit quantization for memory efficiency.
*   **`load_base_model()`**: This function loads the pre-trained language model and its tokenizer using `FastLanguageModel.from_pretrained`. It supports 4-bit quantization via `load_in_4bit=True` to reduce memory footprint.
*   **`attach_lora()`**: This function applies LoRA (Low-Rank Adaptation) adapters to the model. LoRA is a parameter-efficient fine-tuning technique that reduces the number of trainable parameters, making fine-tuning faster and less memory-intensive. It configures the LoRA rank, target modules, alpha, dropout, and uses Unsloth's gradient checkpointing for further VRAM savings.
*   **`cleanup()`**: A utility function to free up GPU memory by calling `gc.collect()` and `torch.cuda.empty_cache()`.

In [ ]:
import gc
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import GRPOConfig, GRPOTrainer

MODEL_ID = "Qwen/Qwen2.5-0.5B"   # swap for any Unsloth-supported model
MAX_SEQ_LEN = 2048                         # Unsloth supports up to ~32k context

# Unsloth turns a lossy 4-bit model back into a usable model via:
#   nf4 quantized weights + bfloat16 compute + double-quant + Flash attention.
# You don't configure bitsandbytes yourself — just flip load_in_4bit=True.
USE_4BIT = True

def load_base_model():
    """Load the base instruct model + tokenizer, with QLoRA 4-bit if enabled."""
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,             # None => auto-pick (bf16 if supported, else fp16)
        load_in_4bit=USE_4BIT,
    )
    return model, tokenizer

def attach_lora(model):
    """Attach LoRA adapters. Unsloth fills in arch-specific target modules
    for you, but for full coverage (attention + MLP) we list them explicitly.
    Returns the LoRA-fitted model (in-place via PEFT)."""
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,                  # LoRA rank. 8/16/32/64; bigger = more capacity + VRAM.
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=32,          # rule of thumb: alpha = 2*r.
        lora_dropout=0.05,
        bias="none",            # train no bias terms.
        use_gradient_checkpointing="unsloth",  # Unsloth's trick: ~30% less VRAM, no slowdown.
        random_state=3407,
        use_rslora=False,      # rsLoRA helps for high-rank; leave off for small r.
        loftq_config=None,
    )
    return model

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
model, tokenizer = load_base_model()
model = attach_lora(model)

# Unsloth's get_chat_template wraps apply_chat_template and maps your
# dataset's keys to the model's template. "chatml" matches Qwen/Llama3.
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",   # also: llama-3, zephyr, mistral, vicuna, alpaca, ...
    map_eos_token=True,
)

==((====))==  Unsloth 2026.9.2: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.9.2 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer_izesd5mv/tokenizer_config.json.


## Import Dataset

Dataset: [`HuggingFaceH4/MATH-500`](https://huggingface.co/datasets/HuggingFaceH4/MATH-500)

Schema:
```
  problem  : the question text (LaTeX)
  solution : the worked solution (LaTeX), ending in \boxed{<answer>}
  answer   : the ground-truth final answer string (e.g. "p - q", "3")
  subject  : e.g. "Precalculus", "Intermediate Algebra"
  level    : 1..5 difficulty
  unique_id: e.g. "test/precalculus/807.json"
```

In [ ]:
from datasets import load_dataset

math500 = load_dataset("HuggingFaceH4/MATH-500", split="test")

SYSTEM = (
    "You are a careful math problem solver. Read the problem, reason step "
    "by step, then give the final answer inside \\boxed{...}."
)

# Prompt-only dataset. `answer` rides along as a hidden column for the
# reward function — it is NOT inserted into the prompt.
grpo_rows = [
    {
        "prompt": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": ex["problem"]},
        ],
        "answer": ex["answer"],   # used only by reward_answer_correctness
    }
    for ex in math500
]
grpo_ds = Dataset.from_list(grpo_rows)

def _extract_last_boxed(text):
    """Pull the contents of the LAST \\boxed{...}, handling one level of
    nested braces (e.g. \\boxed{\\left( 3, \\frac{\\pi}{2} \\right)}).
    A naive `[^{}]*` regex misses ~20% of MATH-500 answers that contain
    nested braces, so the reward would silently score them 0."""
    idx = text.rfind("\\boxed{")
    if idx == -1:
        return ""
    i = idx + len("\\boxed{")
    depth, start = 1, i
    while i < len(text) and depth > 0:
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i].strip()
        i += 1
    return text[start:].strip()

def reward_answer_correctness(completions, **kwargs):
    """+1 if the completion's last \\boxed{...} matches the gold answer,
    else 0. GRPO turns this sparse 0/1 signal into a policy gradient via
    within-group comparison (some generations in the group get it, some
    don't -> the winners reinforce, the losers discourage).

    kwargs carries the per-prompt dataset columns: `answer` is a list
    aligned with `completions` (TRL broadcasts the row's column across
    the num_generations samples for that prompt)."""
    gold_list = kwargs.get("answer", [])
    rewards = []
    # `completions` is expected to be a list of lists of dictionaries, where each inner list
    # contains the `num_generations` for a single prompt, and each dictionary has a 'content' key.
    for i, generations_for_prompt in enumerate(completions):
        gold = str(gold_list[i]).strip() if i < len(gold_list) else ""
        # Iterate through each generated item (dictionary) for the current prompt.
        for comp_dict in generations_for_prompt:
            comp_str = comp_dict['content']
            pred = _extract_last_boxed(comp_str)
            # toy string match — math answer equivalence (e.g. "3" vs "3.0"
            # vs "\\frac{1}{2}") is a hard problem in general. Good enough for
            # a guide reward; use sympy-based normalization for real training.
            rewards.append(1.0 if pred == gold else 0.0)
    return rewards

## GRPO finetuning

This cell configures and runs the Supervised Fine-Tuning (SFT) process:
*   **`GRPOConfig`**: This object defines the training parameters, such as:
    *   `output_dir`: Where to save the fine-tuned model and logs (set to `"out/grpo"`).
    *   `num_train_epochs`: Number of passes over the training data (set to 1).
    *   `per_device_train_batch_size`, `gradient_accumulation_steps`: Control batch size (4) and memory usage (2). Note that `per_device_train_batch_size` must be divisible by `num_generations`.
    *   `num_generations`: Number of completions to generate per prompt (set to 2).
    *   `max_completion_length`: Maximum length for generated completions (set to 512).
    *   `learning_rate`, `lr_scheduler_type`, `warmup_steps`: Learning rate (set to 1e-5), learning rate scheduler type (cosine), and warmup steps (0.05).
    *   `logging_steps`, `save_strategy`: How often to log (every 1 step) and save the model (per epoch).
    *   `bf16`/`fp16`: Enables BFloat16 or Float16 for faster training on compatible GPUs, automatically detected based on hardware support.
    *   `temperature`: Sampling temperature for generation (set to 0.7).
    *   `report_to`: Specifies the reporting backend (set to `"none"`).
*   **`GRPOTrainer`**: Initializes the trainer with the model, tokenizer, training/evaluation datasets, and the configured training arguments.
*   **`grpo_trainer.train()`**: Starts the fine-tuning process. Unsloth patches TRL's `GRPOTrainer` to use its optimized kernels for faster training.

In [ ]:
# CONSTRAINT: per_device_train_batch_size must be divisible by num_generations.
grpo_cfg = GRPOConfig(
    output_dir="out/grpo",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_generations=2,             # completions per prompt -> 2 prompts/step here.
    max_completion_length=512,     # math solutions need room; was 64 for toy QA.
    learning_rate=1e-5,            # GRPO likes an even smaller lr than DPO.
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    logging_steps=1,
    save_strategy="epoch",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    temperature=0.7,               # sampling temperature for generation.
    report_to="none",
)

grpo_trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    args=grpo_cfg,
    train_dataset=grpo_ds,
    reward_funcs=reward_answer_correctness,   # fn OR list of fns.
    # No ref_model, no peft_config — Unsloth handles both.
)

grpo_trainer.train()

Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_answer_correctness / mean,rewards / reward_answer_correctness / std
1,0.000000,0.000000,0.000000,345.125000,19.000000,512.000000,0.500000,178.250000,19.000000,502.000000,0.000015,0.000000,0.000000
2,0.000000,0.000000,0.000000,261.250000,27.000000,512.000000,0.125000,225.428589,27.000000,507.000000,0.000071,0.000000,0.000000
3,0.000000,0.000000,0.000000,315.125000,49.000000,512.000000,0.375000,197.000000,49.000000,502.000000,0.000023,0.000000,0.000000
4,0.000000,0.000000,0.000000,236.625000,18.000000,512.000000,0.250000,144.833344,18.000000,325.000000,0.000022,0.000000,0.000000
5,0.010932,0.125000,0.353553,241.000000,27.000000,512.000000,0.250000,150.666672,27.000000,329.000000,0.000009,0.125000,0.353553
6,0.000000,0.000000,0.000000,294.875000,32.000000,512.000000,0.250000,222.500000,32.000000,491.000000,0.000010,0.000000,0.000000
7,-0.050956,0.125000,0.353553,277.625000,29.000000,512.000000,0.250000,199.500000,29.000000,397.000000,0.000172,0.125000,0.353553
8,0.000000,0.000000,0.000000,487.250000,314.000000,512.000000,0.875000,314.000000,314.000000,314.000000,0.000017,0.000000,0.000000
9,0.000000,0.000000,0.000000,374.625000,6.000000,512.000000,0.375000,292.200012,6.000000,504.000000,0.000044,0.000000,0.000000
10,0.000000,0.000000,0.000000,356.875000,43.000000,512.000000,0.500000,201.750000,43.000000,473.000000,0.000080,0.000000,0.000000


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

TrainOutput(global_step=125, training_loss=0.00010980677265963123, metrics={'train_runtime': 5526.7337, 'train_samples_per_second': 0.09, 'train_steps_per_second': 0.023, 'total_flos': 0.0, 'train_loss': 0.00010980677265963123, 'epoch': 1.0})

In [ ]:
model.save_pretrained("out/grpo_adapter")
tokenizer.save_pretrained("out/grpo_adapter")
print("GRPO(RFT) done -> adapter saved to out/grpo_adapter")

In [ ]:
# Flush memory
del grpo_trainer, model, tokenizer
cleanup()

## Inference

Test the GRPO-finetuned model

In [ ]:
# Load model with trained adapter
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="out/grpo_adapter", # YOUR MODEL YOU TRAINED
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=USE_4BIT,
)

FastLanguageModel.for_inference(model)
model.eval()

In [ ]:
# Prepare a new problem for inference
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "What is the value of $x-2$ if $2x+5=11$?"},
]

# Apply the chat template and tokenize
chat_template_messages = [messages]

inputs = tokenizer(
    [tokenizer.apply_chat_template(x, tokenize=False, add_generation_prompt=True) for x in chat_template_messages],
    return_tensors="pt"
).to("cuda")

# Generate a response
outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)

# Decode and print the output
response = tokenizer.batch_decode(outputs)
print(response[0])